# 02 — Train Stage 2

Run selected V13 experiments on one fixed 80/20 development split. Stage-1 folds are used only to create OOF candidates.


## 1. Project setup


In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

from pathlib import Path
import os
import sys

#PROJECT_DIR = Path('/media/ibmelab/ibme31/YUD/PUMA/Version 13')
PROJECT_DIR = Path('/content/drive/MyDrive/Research/PUMA')

PROJECT_DIR = PROJECT_DIR.expanduser().resolve()
%cd {PROJECT_DIR}

for module_name in list(sys.modules):
    if module_name == 'puma' or module_name.startswith('puma.'):
        del sys.modules[module_name]
if str(PROJECT_DIR) in sys.path:
    sys.path.remove(str(PROJECT_DIR))
sys.path.insert(0, str(PROJECT_DIR))

os.environ.setdefault('PUMA_LORA_GRAD_CHECKPOINTING', '1')
os.environ.setdefault('PUMA_STAGE2_CROP_CACHE_MB', '512')
os.environ.setdefault('PUMA_STAGE2_RESUME_INTERVAL', '5')
os.environ.setdefault('PUMA_V13_AUTO_OOM_FALLBACK', '1')
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
os.environ.setdefault('CUDA_VISIBLE_DEVICES', '0')

print('PROJECT_DIR =', PROJECT_DIR)
print('CUDA_VISIBLE_DEVICES =', os.environ.get('CUDA_VISIBLE_DEVICES'))


In [ ]:
%pip install -q -r requirements_colab.txt

## 2. Runtime


In [ ]:
from puma.runtime import create_runtime, preflight_environment
from puma.stage2.plan import validate_version13_plan

STAGE2_PARALLEL_RUNS = 1
LORA_PARALLEL_RUNS = 1
STAGE2_DATALOADER_WORKERS = 4
FAST_NONDETERMINISTIC = True

runtime = create_runtime(
    PROJECT_DIR,
    run_folds=(0, 1, 2, 3, 4),  # Stage-1 OOF coverage only
    seeds=(0,),
    epochs=60,
    effective_batch_size=256,
    stage2_micro_batch_size=256,
    preprocessing_workers=0,
    early_stopping_enabled=False,
    early_stopping_patience=10,
    early_stopping_min_delta=0.0,
)
runtime.training.number_of_workers = STAGE2_DATALOADER_WORKERS
runtime.training.deterministic = not FAST_NONDETERMINISTIC
runtime.paths.stage2_output_dir = PROJECT_DIR / 'PUMA_stage2_V13_outputs'
runtime.paths.ensure()

print(runtime.as_dict())
preflight_report = preflight_environment(
    runtime, require_dataset=True, require_training_dependencies=True
)


## 3. Fixed Stage-2 split

Create once and reuse for every experiment.


In [ ]:
import pandas as pd
from puma.training.stage2_v13 import ensure_v13_split

REBUILD_SPLIT = False
split_info = ensure_v13_split(
    runtime,
    force=REBUILD_SPLIT,
    val_fraction=0.20,
    seed=2026,
    check_sources=True,
)
print('Split hash:', split_info['split_hash'])
print('Train ROIs:', len(split_info['train_roi_indices']))
print('Validation ROIs:', len(split_info['val_roi_indices']))
print('Case leakage:', split_info['diagnostics']['case_leakage_count'])

summary_csv = runtime.paths.artifact_dir / 'train_val_split' / 'puma_train_val_class_summary.csv'
if summary_csv.exists():
    display(pd.read_csv(summary_csv))


## 4. UNI2-h checkpoint


In [ ]:
from puma.runtime import resolve_hf_token
from puma.models.stage2 import ensure_stage2_pretrained_checkpoints

HF_TOKEN = resolve_hf_token()
checkpoint_summary = ensure_stage2_pretrained_checkpoints(
    PROJECT_DIR, hf_token=HF_TOKEN, pfm_keys=('uni2_h',)
)
print('UNI2-h:', checkpoint_summary['checkpoint_file'])
print('Status:', checkpoint_summary['models']['uni2_h'])


## 5. Stage-1 OOF candidates

All five A1 fold checkpoints are required so every labeled ROI has an out-of-fold Stage-1 prediction.


In [ ]:
import numpy as np
from puma.pipeline.oof import generate_oof_candidates

STAGE1_SEED = 0
missing_checkpoints = []
for fold in range(runtime.data.number_of_folds):
    checkpoint = runtime.paths.stage1_existing_file(
        f'stage1_best_A1_IFCRN_PP_fold{fold}_seed{STAGE1_SEED}.pt'
    )
    if not checkpoint.exists():
        missing_checkpoints.append(str(checkpoint))
if missing_checkpoints:
    raise FileNotFoundError(
        'Missing A1 OOF checkpoint(s):\n' + '\n'.join(missing_checkpoints)
    )

oof_path = generate_oof_candidates(runtime, seed=STAGE1_SEED, force=False)
candidates = np.load(oof_path, mmap_mode='r', allow_pickle=False)
folds, counts = np.unique(candidates['fold'], return_counts=True)
print('OOF candidates by fold:', dict(zip(folds.tolist(), counts.tolist())))
assert set(folds.tolist()) == set(range(runtime.data.number_of_folds))


## 6. Select experiments

Remove names from the list to run a smaller queue. All selected experiments use the same split.


In [ ]:
from puma.stage2.catalog import VERSION13_EXPERIMENT_PURPOSE
from puma.stage2.plan import validate_version13_plan
from puma.pipeline.experiments_v13 import v13_experiment_status

STAGE2_EXPERIMENTS_TO_RUN = [
    'V13_01_META_NEW_SPLIT_FROZEN',
    'V13_02_META_CONTEXT_NEW_SPLIT_FROZEN',
    'V13_03_META_CONTEXT_CBFOCAL_FROZEN',
    #'V13_04_META_CONTEXT_CBCE_FROZEN',
    #'V13_05_META_CONTEXT_RAREBOOST_FROZEN',
    #'V13_06_META_CONTEXT_LORA_R8_B4',
]

plan = validate_version13_plan(
    runtime,
    STAGE2_EXPERIMENTS_TO_RUN,
    parallel_runs=STAGE2_PARALLEL_RUNS,
    lora_parallel_runs=LORA_PARALLEL_RUNS,
)
print('Plan:', plan)
for name in STAGE2_EXPERIMENTS_TO_RUN:
    print(f'- {name}: {VERSION13_EXPERIMENT_PURPOSE[name]}')
display(v13_experiment_status(runtime))


In [ ]:
from puma.stage2.catalog import VERSION13_EXPERIMENT_PURPOSE
from puma.stage2.plan import validate_version13_plan
from puma.pipeline.experiments_v13 import v13_experiment_status

STAGE2_EXPERIMENTS_TO_RUN = [
    #'V13_01_META_NEW_SPLIT_FROZEN',
    #'V13_02_META_CONTEXT_NEW_SPLIT_FROZEN',
    #'V13_03_META_CONTEXT_CBFOCAL_FROZEN',
    'V13_04_META_CONTEXT_CBCE_FROZEN',
    'V13_05_META_CONTEXT_RAREBOOST_FROZEN',
    'V13_06_META_CONTEXT_LORA_R8_B4',
]

plan = validate_version13_plan(
    runtime,
    STAGE2_EXPERIMENTS_TO_RUN,
    parallel_runs=STAGE2_PARALLEL_RUNS,
    lora_parallel_runs=LORA_PARALLEL_RUNS,
)
print('Plan:', plan)
for name in STAGE2_EXPERIMENTS_TO_RUN:
    print(f'- {name}: {VERSION13_EXPERIMENT_PURPOSE[name]}')
display(v13_experiment_status(runtime))


## 7. Train

`STAGE2_PARALLEL_RUNS=1` is the safe default for one RTX 3090.


In [ ]:
from puma.pipeline.experiments_v13 import run_stage2_v13_program

stage2_run_summary = run_stage2_v13_program(
    runtime,
    hf_token=HF_TOKEN,
    experiments_to_run=STAGE2_EXPERIMENTS_TO_RUN,
    parallel_runs=STAGE2_PARALLEL_RUNS,
    lora_parallel_runs=LORA_PARALLEL_RUNS,
)
stage2_run_summary


## 8. Results


In [ ]:
from puma.pipeline.experiments_v13 import aggregate_v13_results, v13_experiment_status

ranking = aggregate_v13_results(runtime, tuple(STAGE2_EXPERIMENTS_TO_RUN))
if ranking.empty:
    print('No completed V13 experiment yet.')
else:
    sort_columns = [
        c for c in ('macro_f1', 'conditional_type_macro_f1_present', 'reject_f1')
        if c in ranking.columns
    ]
    display(ranking.sort_values(sort_columns, ascending=False, na_position='last'))
display(v13_experiment_status(runtime))


## 9. Lock the winner

Lock only after reviewing the completed experiments.


In [ ]:
LOCK_BEST_AFTER_REVIEW = False
SELECTED_EXPERIMENT = None  # set a name, or keep None to auto-rank the selected queue

if LOCK_BEST_AFTER_REVIEW:
    from puma.pipeline.experiments_v13 import lock_v13_winner

    locked = lock_v13_winner(
        runtime,
        selected_experiment=SELECTED_EXPERIMENT,
        candidate_experiments=STAGE2_EXPERIMENTS_TO_RUN,
    )
    print('Locked winner:', locked['selected_experiment'])
    print('Final plan:', locked['final_training_plan'])
    locked
